In [1]:
#!/usr/bin/env python3
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
DATA_DIR   = Path("smb_data")  # Input CSVs directory
ART_DIR    = Path("artifacts") # Output artifacts directory
ART_DIR.mkdir(exist_ok=True)


# MODELS & ALGOs

In [ ]:
#!/usr/bin/env python3
# -----------------------------------------------------------------------
# ICE-ID ER Runner: TriBERTa, DistilBERT, paraphrase-MiniLM
# multiprocessing, batching, 10 runs with random pair sampling
# -----------------------------------------------------------------------

import itertools
import logging
import gc
import random
from pathlib import Path
from multiprocessing import Pool, cpu_count

import networkx as nx
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    accuracy_score,
    roc_curve
)
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder,
    InputExample,
    losses
)

# ────────────────────────────────────────────────────────────────
# CONFIG
# ────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(message)s",
                    datefmt="%H:%M:%S")
ART_DIR = Path("artifacts")
DATA_DIR = Path("raw_data")
OUTPUT   = Path("models_er");  OUTPUT.mkdir(exist_ok=True)
PLOTS    = OUTPUT / "plots";   PLOTS.mkdir(exist_ok=True)
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_RUNS           = 10
MAX_TRAIN_PAIRS  = 200_000
MAX_TEST_PAIRS   = 200_000
TRIPLET_SAMPLES  = 1_000
BATCH_SIZE_EMB   = 1_024
DISTIL_BATCH     = 32
MINILM_BATCH     = 32
RNG_SEED_BASE    = 42

# ────────────────────────────────────────────────────────────────
# LOAD DATA
# ────────────────────────────────────────────────────────────────
def load_people():
    ppl = pd.read_csv(DATA_DIR / "people.csv", low_memory=False).rename(columns=str.lower)
    if "id" not in ppl.columns:
        raise KeyError("`id` column missing")
    ppl = ppl.set_index("id")
    for col in ["first_name", "middle_name", "patronym", "surname"]:
        ppl[col] = ppl.get(col, "").fillna("").astype(str)
    ppl["full_name"] = (
        ppl[["first_name", "middle_name", "patronym", "surname"]]
        .apply(lambda r: " ".join(w.strip().lower() for w in r if w), axis=1)
    )
    ppl["birthyear"] = pd.to_numeric(ppl.get("birthyear", 0),
                                     errors="coerce").fillna(0).astype(int)
    ppl["heimild"]   = pd.to_numeric(ppl.get("heimild", 0),
                                     errors="coerce").fillna(0).astype(int)
    lbl = (pd.read_csv(ART_DIR / "row_labels.csv")
             .set_index("row_id")["person"]
             .pipe(pd.to_numeric, errors="coerce")
             .astype("Int64"))
    ppl["person"] = pd.Series(ppl.index, index=ppl.index).map(lbl)
    return ppl.reset_index()

# ────────────────────────────────────────────────────────────────
# BLOCKING
# ────────────────────────────────────────────────────────────────
def trigram(s):
    s = "".join(c for c in s.lower() if c.isalnum())
    return {s[i:i+3] for i in range(len(s)-2)} if len(s) >= 3 else {s}

def build_blocks(df):
    with Pool(cpu_count()) as p:
        trigs = p.map(trigram, df["full_name"].tolist())
    df = df.copy(); df["trigs"] = trigs
    blocks = {}
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building blocks"):
        key = (frozenset(sorted(row["trigs"])),
               (row["birthyear"]//10) if row["birthyear"]>0 else -1)
        blocks.setdefault(key, []).append(idx)
    return blocks

# ────────────────────────────────────────────────────────────────
# PAIR SAMPLING
# ────────────────────────────────────────────────────────────────
def _sample_pairs_from_bucket(bucket, k, rng):
    n = len(bucket)
    if n<2: return []
    max_pairs = n*(n-1)//2
    k = min(k, max_pairs)
    if max_pairs <= 3000:
        pairs = list(itertools.combinations(bucket,2))
        return rng.sample(pairs,k)
    res = set()
    while len(res)<k:
        a,b = rng.sample(bucket,2)
        res.add(tuple(sorted((a,b))))
    return list(res)

def sample_pairs(blocks, train_idx, test_idx, max_train, max_test, rng):
    p_tr,p_te = [],[]
    buckets = list(blocks.values()); rng.shuffle(buckets)
    for b in buckets:
        if len(p_tr)<max_train:
            tb = [i for i in b if i in train_idx]
            p_tr += _sample_pairs_from_bucket(tb, max_train-len(p_tr), rng)
        if len(p_te)<max_test:
            tb = [i for i in b if i in test_idx]
            p_te += _sample_pairs_from_bucket(tb, max_test-len(p_te), rng)
        if len(p_tr)>=max_train and len(p_te)>=max_test:
            break
    return p_tr,p_te

# ────────────────────────────────────────────────────────────────
# LABELS
# ────────────────────────────────────────────────────────────────
def make_labels(pairs, df):
    return np.array([int(df.at[i,"person"]==df.at[j,"person"] and not pd.isna(df.at[i,"person"]))
                     for i,j in pairs], dtype=int)

# ────────────────────────────────────────────────────────────────
# MODELS
# ────────────────────────────────────────────────────────────────
class TriBERTaER:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=BATCH_SIZE_EMB):
        self.model = SentenceTransformer(model_name, device=DEVICE)
        self.loss  = losses.TripletLoss(self.model)
        self.batch_size = batch_size
    @staticmethod
    def serial(row):
        return " ; ".join(f"{k}: {v}" for k,v in row.items())
    def train(self, df, train_idx, num_triplets=TRIPLET_SAMPLES):
        examples=[]; grouped=df.loc[train_idx].dropna(subset=["person"]).groupby("person")
        persons=list(grouped.groups); rng=random.Random(RNG_SEED_BASE)
        while len(examples)<num_triplets:
            pid=rng.choice(persons)
            m=grouped.get_group(pid).index.tolist()
            if len(m)<2: continue
            a,p=rng.sample(m,2)
            neg=rng.choice([x for x in persons if x!=pid])
            n=rng.choice(grouped.get_group(neg).index.tolist())
            examples.append(InputExample(texts=[self.serial(df.loc[a]),
                                                self.serial(df.loc[p]),
                                                self.serial(df.loc[n])]))
        self.model.fit(train_objectives=[(examples,self.loss)], epochs=1, train_batch_size=16, use_amp=True)
    def score(self, df, pairs):
        sims=[]
        for i in range(0,len(pairs),self.batch_size):
            batch=pairs[i:i+self.batch_size]
            a=[self.serial(df.loc[a]) for a,_ in batch]
            b=[self.serial(df.loc[b]) for _,b in batch]
            ea=self.model.encode(a,convert_to_tensor=True,device=DEVICE)
            eb=self.model.encode(b,convert_to_tensor=True,device=DEVICE)
            sims.append(torch.cosine_similarity(ea,eb).cpu().numpy())
        return np.concatenate(sims)/2+0.5

class DistilBERTER:
    def __init__(self, model_id="distilbert-base-uncased", batch_size=DISTIL_BATCH):
        self.tok=AutoTokenizer.from_pretrained(model_id)
        self.model=AutoModelForSequenceClassification.from_pretrained(model_id,num_labels=2).to(DEVICE)
        self.batch_size=batch_size
    def train(self, df, pairs, labels):
        texts=[f"A: {TriBERTaER.serial(df.loc[i])}\nB: {TriBERTaER.serial(df.loc[j])}" for i,j in pairs]
        enc=self.tok(texts,truncation=True,padding=True,max_length=256,return_tensors="pt")
        ds=torch.utils.data.TensorDataset(enc["input_ids"],enc["attention_mask"],torch.tensor(labels))
        args=TrainingArguments(output_dir=OUTPUT/"distil_tmp",per_device_train_batch_size=8,
                               num_train_epochs=1,logging_steps=100,save_strategy="no",
                               fp16=True,remove_unused_columns=False)
        Trainer(model=self.model,args=args,train_dataset=ds,tokenizer=self.tok).train()
    def score(self, df, pairs):
        texts=[f"A: {TriBERTaER.serial(df.loc[i])}\nB: {TriBERTaER.serial(df.loc[j])}" for i,j in pairs]
        probs=[]
        for i in range(0,len(texts),self.batch_size):
            batch=texts[i:i+self.batch_size]
            enc=self.tok(batch,truncation=True,padding=True,max_length=256,return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                lg=self.model(**enc).logits
            probs.append(torch.softmax(lg,-1)[:,1].cpu().numpy())
        return np.concatenate(probs)

class MiniLMCE:
    def __init__(self, model_id="sentence-transformers/paraphrase-MiniLM-L6-v2", batch_size=MINILM_BATCH):
        self.model=CrossEncoder(model_id,num_labels=2,device=DEVICE)
        self.batch_size=batch_size
    def train(self, df, pairs, labels):
        ex=[InputExample(texts=[TriBERTaER.serial(df.loc[i]),TriBERTaER.serial(df.loc[j])],
                        label=float(l)) for (i,j),l in zip(pairs,labels)]
        self.model.fit(train_dataloader=ex,epochs=1,batch_size=16)
    def score(self, df, pairs):
        texts=[(TriBERTaER.serial(df.loc[i]),TriBERTaER.serial(df.loc[j])) for i,j in pairs]
        return np.array(self.model.predict(texts,batch_size=self.batch_size))

# ────────────────────────────────────────────────────────────────
# METRICS & UTILITIES
# ────────────────────────────────────────────────────────────────
def bin_metrics(y,p,thr=0.5):
    pred=(p>=thr).astype(int)
    pr,rc,f1,_=precision_recall_fscore_support(y,pred,average="binary",zero_division=0)
    acc=accuracy_score(y,pred)
    auc=roc_auc_score(y,p) if len(np.unique(y))>1 else float("nan")
    return {"precision":pr,"recall":rc,"f1":f1,"accuracy":acc,"auc":auc}

def save_summary(s,fn):
    pd.DataFrame(s).T.to_csv(OUTPUT/fn)

def cluster_and_save(df,pairs,probs,thr=0.5):
    G=nx.Graph(); G.add_nodes_from(df.index)
    for (i,j),p in tqdm(zip(pairs,probs),total=len(pairs),desc="Clustering"):
        if p>=thr: G.add_edge(i,j)
    comp={n:cid for cid,comp in enumerate(nx.connected_components(G)) for n in comp}
    df["cluster_id"]=df.index.map(comp)
    df.to_csv(OUTPUT/"er_clusters.csv",index=False)

# ────────────────────────────────────────────────────────────────
# MAIN
# ────────────────────────────────────────────────────────────────
if __name__=="__main__":
    df=load_people()
    df_lab=df[df["person"].notna()].reset_index(drop=True)
    blocks=build_blocks(df_lab)
    run_metrics=[]
    for run in range(N_RUNS):
        rng=random.Random(RNG_SEED_BASE+run)
        tr,te=train_test_split(df_lab.index.tolist(),test_size=0.2,random_state=run)
        p_tr,p_te=sample_pairs(blocks,tr,te,MAX_TRAIN_PAIRS,MAX_TEST_PAIRS,rng)
        y_tr, y_te = make_labels(p_tr,df_lab), make_labels(p_te,df_lab)
        tb,db,ce=TriBERTaER(),DistilBERTER(),MiniLMCE()
        tb.train(df_lab,tr);   p_tb=tb.score(df_lab,p_te)
        db.train(df_lab,p_tr,y_tr); p_db=db.score(df_lab,p_te)
        ce.train(df_lab,p_tr,y_tr); p_ce=ce.score(df_lab,p_te)
        run_metrics.append({
            "TriBERTa":bin_metrics(y_te,p_tb),
            "DistilBERT":bin_metrics(y_te,p_db),
            "MiniLM-CE":bin_metrics(y_te,p_ce)
        })
        del tb,db,ce; torch.cuda.empty_cache(); gc.collect()
    models=["TriBERTa","DistilBERT","MiniLM-CE"]
    keys=["precision","recall","f1","accuracy","auc"]
    avg={m:{k:np.mean([r[m][k] for r in run_metrics]) for k in keys} for m in models}
    save_summary(avg,"metrics_summary_avg.csv")
    cluster_and_save(df_lab,p_te,p_tb,thr=0.5)
    logging.info("Done.")

Building blocks:   0%|          | 0/476683 [00:00<?, ?it/s]

17:03:23 INFO Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
17:03:25 WARNING Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


OSError: paraphrase-MiniLM-L6-v2 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`